# Content-Based Filtering

### Data Loading

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('data/newMusicInfo.csv')
df.head()

,track_id,name,artist,spotify_preview_url,tags,genre,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,TROUVHL128F426C441,Come as You Are,NIRVANA,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, GRUNGE",RNB,1991,218920,0.508,0.826,4,-5.783,0,0.0400,0.000175,0.000459,0.0878,0.543,120.012,4
1,TRLNZBD128F935E4D8,Creep,RADIOHEAD,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...,"ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...",RNB,2008,238640,0.515,0.430,7,-9.935,1,0.0369,0.010200,0.000141,0.1290,0.104,91.841,4
2,TRMZXEW128F9341FD5,The Scientist,COLDPLAY,https://p.scdn.co/mp3-preview/95cb9df1b056d759...,"ROCK, ALTERNATIVE, INDIE, POP, ALTERNATIVE_ROC...",ROCK,2007,311014,0.566,0.429,5,-7.826,1,0.0242,0.715000,0.000014,0.1200,0.173,146.365,4
3,TRIODZU128E078F3E2,Under the Bridge,RED HOT CHILI PEPPERS,https://p.scdn.co/mp3-preview/90e41778392f27b6...,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, FUNK",POP,2003,265506,0.554,0.490,4,-8.046,1,0.0457,0.016800,0.000534,0.1360,0.513,84.275,4
4,TRYEZGE128F92D04C8,Feel Good Inc.,GORILLAZ,https://p.scdn.co/mp3-preview/7388c425022ced92...,"ROCK, ELECTRONIC, ALTERNATIVE, INDIE, POP, ALT...",RNB,2010,223106,0.809,0.720,6,-7.776,1,0.1360,0.033700,0.001410,0.8260,0.832,138.556,4


### Selected only the relevant features

In [3]:
df_selected = df[['track_id', 'name','artist', 'tags', 'genre', 'year', 'danceability', 'speechiness', 'tempo','spotify_preview_url','duration_ms']]
df_selected.head()

,track_id,name,artist,tags,genre,year,danceability,speechiness,tempo,spotify_preview_url,duration_ms
0,TROUVHL128F426C441,Come as You Are,NIRVANA,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, GRUNGE",RNB,1991,0.508,0.0400,120.012,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...,218920
1,TRLNZBD128F935E4D8,Creep,RADIOHEAD,"ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...",RNB,2008,0.515,0.0369,91.841,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...,238640
2,TRMZXEW128F9341FD5,The Scientist,COLDPLAY,"ROCK, ALTERNATIVE, INDIE, POP, ALTERNATIVE_ROC...",ROCK,2007,0.566,0.0242,146.365,https://p.scdn.co/mp3-preview/95cb9df1b056d759...,311014
3,TRIODZU128E078F3E2,Under the Bridge,RED HOT CHILI PEPPERS,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, FUNK",POP,2003,0.554,0.0457,84.275,https://p.scdn.co/mp3-preview/90e41778392f27b6...,265506
4,TRYEZGE128F92D04C8,Feel Good Inc.,GORILLAZ,"ROCK, ELECTRONIC, ALTERNATIVE, INDIE, POP, ALT...",RNB,2010,0.809,0.1360,138.556,https://p.scdn.co/mp3-preview/7388c425022ced92...,223106


### Check data type

In [4]:
df_selected.dtypes

track_id                object
name                    object
artist                  object
tags                    object
genre                   object
year                     int64
danceability           float64
speechiness            float64
tempo                  float64
spotify_preview_url     object
duration_ms              int64
dtype: object

### Copy and clean

In [5]:
df_selected = df[['track_id', 'name', 'artist', 'tags', 'genre', 'year', 'danceability', 'speechiness', 'tempo','spotify_preview_url','duration_ms']].copy()
df_selected[['name', 'artist', 'tags', 'genre']] = df_selected[['name', 'artist', 'tags', 'genre']].fillna('')

### Combine text features

In [6]:
df_selected['combined_text'] = df_selected['tags'] + ' ' + df_selected['genre'] + ' ' + df_selected['artist']

### TF-IDF on combined text

In [7]:
# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(
    analyzer='word', 
    ngram_range=(1, 2),  # Include unigrams and bigrams
    min_df=1, # Keep words that appear in at least 1 document
    stop_words='english'
)

# Transform the combined text column into TF-IDF features
text_matrix = tfidf.fit_transform(df_selected['combined_text'])

### Normalize numeric features

In [8]:
# Initialize and apply MinMaxScaler to scale all numeric columns to [0, 1]
scaler = MinMaxScaler()
# Apply scaling to numerical features
numeric_matrix = scaler.fit_transform(df_selected[['danceability', 'speechiness', 'tempo', 'year']])

### Combine all features

In [9]:
combined_matrix = hstack([text_matrix, numeric_matrix]).tocsr()

### Fit KNN model

In [10]:
knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
knn_model.fit(combined_matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

### Build name, tag, genre, year and index mapping

In [11]:
# lowercase song names in mapping
# Build the (song name) -> index map
name_to_index = {
    row['name'].lower().strip(): idx
    for idx, row in df_selected.iterrows()
}

In [12]:
# Create a dictionary to map (tag, genre, year) -> list of row indices
tag_genre_year_to_indices = {}

# Loop through each row in the dataframe
for idx, row in df_selected.iterrows():
    # Build a unique key using tag, genre, and year (all lowercase for consistency)
    key = (row['tags'].lower().strip(), row['genre'].lower().strip(), row['year'])
    
    # Insert the key into the dictionary if it doesn't exist, and append the current row index to its list
    tag_genre_year_to_indices.setdefault(key, []).append(idx)

In [13]:
# Create a dictionary to map (genre, year) -> list of row indices
genre_year_to_indices = {}

# Loop through each row in the dataframe
for idx, row in df_selected.iterrows():
    # Build a unique key using genre and year (all lowercase for consistency)
    key = (row['genre'].lower().strip(), row['year'])
    
     # Insert the key into the dictionary if it doesn't exist, and append the current row index to its list
    genre_year_to_indices.setdefault(key, []).append(idx)

In [14]:
genre_to_indices = {}

# Loop through each row in the dataframe
for idx, row in df_selected.iterrows():
    # Build a unique key using genre (all lowercase for consistency)
    key = str(row['genre']).lower().strip()
    
     # Insert the key into the dictionary if it doesn't exist, and append the current row index to its list
    genre_to_indices.setdefault(key, []).append(idx)

### Recommendation function using name

##### convert time ms to min

In [15]:
def ms_to_minutes_seconds(ms):
    try:
        ms = int(ms)
        seconds = ms // 1000
        minutes = seconds // 60
        remaining_seconds = seconds % 60
        return f"{minutes}:{remaining_seconds:02d}"
    except Exception:
        return None

##### Calculation accuracy function

In [16]:
def calculate_accuracy_from_distance(distances):
    # Convert KNN distances into similarity percentages.
    return [(1 - d) * 100 for d in distances]

##### Validation for tempo, danceability and year

In [17]:
def validate_song_name(song_name):
    # Validate that the song exists in the dataset
    song_key = song_name.lower().strip()
    if song_key not in name_to_index:
        return None, f"Song '{song_name}' not found in the dataset."
    return song_key, None

##### Main Recommendation using name Function

In [18]:
def recommend_songs_by_name(song_name, n_recommendations=4):
    # Preprocess the input name (normalize: lowercase + remove spaces)
    song_key = song_name.lower().strip()
    
    # 1 - Check if the song exists in the mapping
    song_key, error = validate_song_name(song_name)
    if error:
        return error
    
    # 2 - Get the index of the input song directly from the dictionary
    idx = name_to_index[song_key] 
    
    # 3 - Find nearest neighbors using the KNN model
    distances, indices = knn_model.kneighbors(
        combined_matrix[idx], n_neighbors=n_recommendations + 1
    )
    
    # 4 - Exclude the first result (it’s always the song itself)
    recommended_indices = indices.flatten()[1:]
    recommended_distances = distances.flatten()[1:]
    
    # 5 - Get details of the input song
    input_song = df_selected.iloc[[idx]][
        ['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']
    ]
    
    # 6 - Get details of recommended songs
    recommendations = df_selected.iloc[recommended_indices][
        ['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']
    ]
       
    
    # 7 - Add similarity score ("accuracy") column and calculation 
    input_song = input_song.assign(accuracy=["100%"])
    similarities = calculate_accuracy_from_distance(recommended_distances)
    recommendations = recommendations.assign(
        accuracy=[f"{s:.2f}%" for s in similarities]
    )
    
    # 8 - Merge input + recommendations into one DataFrame
    result = pd.concat([input_song, recommendations]).reset_index(drop=True)
    
    # 9 - convert time ms to min
    result['duration'] = result['duration_ms'].apply(ms_to_minutes_seconds)
    result = result.drop(columns=['duration_ms'])
    
    return result

### Recommendation function using genre, tags, tempo and danceability

##### Calculation accuracy function

In [19]:
def calculate_accuracy(song_tempo, song_dance, input_tempo, input_dance):
    
    # Calculate similarity with input manually (tempo + danceability diff)
    tempo_diff = abs(song_tempo - input_tempo) / max(input_tempo, 1)
    dance_diff = abs(song_dance - input_dance)
    
    # Simple similarity formula: 1 - take 50% weight from tempo difference + 50% from danceability difference.
    sim = (1 - (0.5 * tempo_diff + 0.5 * dance_diff)) * 100
    
    # Clip the score between 0 and 100 to avoid negative or >100 values.
    sim = max(0, min(100, sim)) 
    
    return f"{sim:.2f}%"

##### Validation for tempo, danceability and year

In [20]:
def validate_inputs(song_tempo, danceability_value, year):
    
    # Validate tempo (must be numeric)
    try:
        song_tempo = float(song_tempo)
    except ValueError:
        return "Tempo must be a number (BPM)."

    # Validate danceability (0.0–1.0)
    try:
        danceability_value = float(danceability_value)
    except ValueError:
        return "Danceability must be between 0.0 and 1.0."
    if not (0.0 <= danceability_value <= 1.0):
        return "Danceability must be between 0.0 and 1.0."

    # Validate year (must be integer)
    try:
        year = int(year)
    except ValueError:
        return "Year must be an integer."

    return song_tempo, danceability_value, year


##### Main Recommendation using genre, tags, tempo and danceability Function

In [21]:
def recommend_songs_by_genre_tempo_tags_danceability(
    song_genre, song_tempo, song_tags, danceability_value, year, n_recommendations=4
):
    # Normalize text fields (remove spaces + lowercase for consistency)
    song_genre = song_genre.lower().strip()
    song_tags = song_tags.lower().strip()

    # Run input validation
    validated = validate_inputs(song_tempo, danceability_value, year)
    
    # error message returned
    if isinstance(validated, str):  
        return validated 
    song_tempo, danceability_value, year = validated

    # 1 - fallback search
    candidates = None
    key_tag = (song_tags, song_genre, year)
    
    # First try - (tag + genre + year) exact match
    if key_tag in tag_genre_year_to_indices:
        candidates = df_selected.iloc[tag_genre_year_to_indices[key_tag]]

    # Second try - (genre + year)
    elif (song_genre, year) in genre_year_to_indices:
        candidates = df_selected.iloc[genre_year_to_indices[(song_genre, year)]]

    # Third try - only genre
    elif song_genre in genre_to_indices:
        candidates = df_selected.iloc[genre_to_indices[song_genre]]

    # If nothing matches - return not found message
    if candidates is None or candidates.empty:
        return f"No songs found in genre '{song_genre}' for year {year} (with/without tag '{song_tags}')."

    # 2 - Tempo filter ±5
    tempo_matches = candidates[(candidates['tempo'] >= song_tempo - 5) & (candidates['tempo'] <= song_tempo + 5)]
    if tempo_matches.empty:
        return f"No songs in genre '{song_genre}' near {song_tempo} BPM."

    # 3 - Danceability filter ±0.1
    final_matches = tempo_matches[
        (tempo_matches['danceability'] >= danceability_value - 0.1) & 
        (tempo_matches['danceability'] <= danceability_value + 0.1)
    ].copy()
    if final_matches.empty:
        return f"No songs in genre '{song_genre}' near {song_tempo} BPM with danceability {danceability_value}."

    # 4 - Pick the "closest" candidate (min absolute difference in danceability)
    final_matches['diff'] = (
        (final_matches['tempo'] - song_tempo).abs() +
        (final_matches['danceability'] - danceability_value).abs() + 
        (final_matches['year'] - year).abs()
    )
    idx = final_matches['diff'].idxmin()

    # 5 - Run KNN
    distances,indices = knn_model.kneighbors(combined_matrix[idx], n_neighbors=n_recommendations + 1)
    recommended_indices = indices.flatten()[1:]
    # recommended_distances = distances.flatten()[1:]

    # 6 - Get details of the input song
    input_song = df_selected.iloc[[idx]][['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']]
    
    # Check if input exactly matches the chosen candidate (strict match on 4 fields)
    is_exact_match = (
        input_song['genre'].iloc[0].lower().strip() == song_genre and
        input_song['tags'].iloc[0].lower().strip() == song_tags and
        input_song['year'].iloc[0] == year and
        abs(input_song['tempo'].iloc[0] - song_tempo) < 1e-6 and
        abs(input_song['danceability'].iloc[0] - danceability_value) < 1e-6
    )
    
    # 7 - Get details of recommended songs
    recommendations = df_selected.iloc[recommended_indices][['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']]

    # 8 - Add similarity score ("accuracy") column and calculation 
    if is_exact_match:
        input_song = input_song.assign(accuracy=["100%"])
    else:
        ## Input song accuracy     
        acc = calculate_accuracy(
            input_song['tempo'].iloc[0],
            input_song['danceability'].iloc[0],
            song_tempo,
            danceability_value
        )
        input_song = input_song.assign(accuracy=[acc])
        
    ## Recommended songs accuracy
    # For each recommended song, calculate similarity to the user’s input values.
    recommendations = df_selected.iloc[recommended_indices][['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']].copy()
    recommendations['accuracy_val'] = recommendations.apply(
        lambda row: float(calculate_accuracy(row['tempo'], row['danceability'], song_tempo, danceability_value).replace("%","")),
        axis=1
    )
    
    # Sort by accuracy high → low
    recommendations = recommendations.sort_values(by="accuracy_val", ascending=False)
    
    # Format accuracy
    recommendations['accuracy'] = recommendations['accuracy_val'].map(lambda s: f"{s:.2f}%")
    recommendations = recommendations.drop(columns="accuracy_val")

    # 9 - Merge input + recommendations into one DataFrame
    result =  pd.concat([input_song, recommendations]).reset_index(drop=True)
    
    # 10 - convert time ms to min
    result['duration'] = result['duration_ms'].apply(ms_to_minutes_seconds)
    result = result.drop(columns=['duration_ms'])
    
    return result


###  User input

In [22]:
song_input = input("Enter a song name: ").strip()

recommendations = recommend_songs_by_name(song_input, n_recommendations=4)
print("\n Recommended Songs:\n", recommendations)
# print("\n Hidden Info:\n", extra)


Enter a song name:  love



 Recommended Songs:
              track_id              name                 artist  year  \
0  TROHCFR128F92DDDA9              Love  THE SMASHING PUMPKINS  1995   
1  TRZLXIO128F931C220        Panopticon  THE SMASHING PUMPKINS  2012   
2  TRIEODF128E0788AA5  Appels + Oranjes  THE SMASHING PUMPKINS  1998   
3  TRMDJVM128E078907A         Tristessa  THE SMASHING PUMPKINS  1991   
4  TRFORRG128F426AC93              Luna  THE SMASHING PUMPKINS  1993   

        genre    tempo                                               tags  \
0  ELECTRONIC   95.210  ROCK, ELECTRONIC, ALTERNATIVE, ALTERNATIVE_ROC...   
1  ELECTRONIC  100.639          ROCK, ALTERNATIVE_ROCK, HARD_ROCK, GRUNGE   
2        ROCK  112.824  ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, AMER...   
3        ROCK  103.059  ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, AMER...   
4        ROCK  125.415  ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, PSYC...   

   danceability                                spotify_preview_url accuracy  \
0  

In [23]:
genre_input = input("Enter a genre separated by commas (e.g. Rock, electronic, Pop): ").strip()
tag_input = input("Enter a tag (e.g. indie, folk, love): ").strip()
year_input = input("Enter year (e.g. 1900 - 2022): ").strip()
tempo_input = input("Enter a tempo (BPM) (0-239): ").strip()
danceability_input = input("Enter danceability (0.0 to 1.0): ").strip()

recommendations = recommend_songs_by_genre_tempo_tags_danceability(song_genre=genre_input,song_tempo=tempo_input,song_tags=tag_input,danceability_value=danceability_input,year=year_input,n_recommendations=4)
print("\n🎶 Recommended Songs:\n", recommendations)

Enter a genre separated by commas (e.g. Rock, electronic, Pop):  RNB
Enter a tag (e.g. indie, folk, love):  Beautiful
Enter year (e.g. 1900 - 2022):  2020
Enter a tempo (BPM) (0-239):  120
Enter danceability (0.0 to 1.0):  0.50



🎶 Recommended Songs:
              track_id         name     artist  year       genre    tempo  \
0  TRFOBLN128E0785ED9  Deutschland  RAMMSTEIN  2019         RNB  120.117   
1  TRVBEOH128F9305C8B     Haifisch  RAMMSTEIN  2009        ROCK  134.980   
2  TRAQFSP128F9308509      Seemann  RAMMSTEIN  1995       METAL  140.150   
3  TRIMATH128F14A0C1B        Nebel  RAMMSTEIN  2001  ELECTRONIC  102.104   
4  TRZMLSV128F9305C8C    B********  RAMMSTEIN  2009        ROCK  141.016   

                                     tags  danceability  \
0               METAL, INDUSTRIAL, GERMAN         0.521   
1  METAL, HEAVY_METAL, INDUSTRIAL, GERMAN         0.627   
2               METAL, INDUSTRIAL, GERMAN         0.600   
3               METAL, INDUSTRIAL, GERMAN         0.628   
4               METAL, INDUSTRIAL, GERMAN         0.352   

                                 spotify_preview_url accuracy duration  
0  https://p.scdn.co/mp3-preview/a16ac8b56d54533f...   98.90%     5:22  
1  https://p.scdn.c